In [0]:
data = [
 ("U001", "2024-01-01 10:00:00"),
 ("U001", "2024-01-01 10:10:00"),
 ("U001", "2024-01-01 11:00:00"),
 ("U001", "2024-01-01 11:05:00"),
 ("U002", "2024-01-01 12:00:00"),
 ("U002", "2024-01-01 13:00:00"),
 ("U003", "2024-01-01 09:00:00"),
 ("U003", "2024-01-01 09:35:00"),
]

#Create DataFrame
df = spark.createDataFrame(data, ["user_id", "timestamp"])

In [0]:
display(df)

In [0]:
df.dtypes

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

df = df.withColumn('timestamp',col('timestamp').cast('timestamp'))

window_spec = Window.partitionBy('user_id').orderBy('timestamp')

df_mindates = df.withColumn('min_date',min('timestamp').over(window_spec))\
    .withColumn('date_diff_minutes',(unix_timestamp('timestamp')-unix_timestamp('min_date'))/60)\
    .withColumn('prev_diff',lag('date_diff_minutes').over(window_spec))\
    .withColumn('session_time',(col('date_diff_minutes')-col('prev_diff')).cast('int'))\
    .withColumn('new_session',when((col('session_time')>30) | (col('session_time').isNull()) ,1).otherwise(0))\
    .withColumn('session_id',sum('new_session').over(window_spec))

display(df_mindates)

In [0]:
df_result = df_mindates.groupBy('user_id','session_id').agg(
    min("timestamp").alias('session_start_date'),
    max("timestamp").alias('session_end_date'),
    count('*').alias('session_activity_counts')
)
display(df_result)